# Staged Composite Analysis for PT Precast Plank + In-situ Topping

This notebook is set up for your **PS4A plank**:

- precast plank width = **2400 mm**
- precast plank thickness = **110 mm**
- in-situ topping thickness = **115 mm**
- total composite depth = **225 mm**
- span = **4499 mm**
- prestressing = **8 × 9.6 mm strands**
- strand cover to strand centroid = **25 mm from soffit** *(edit if your detail means something else)*

It uses **staged construction**:

1. **Stage 1 — precast only**  
   Prestressed plank carries its own self-weight, wet topping, and construction load *(unless propped)*.

2. **Stage 2 — composite section**  
   After topping hardens and interface shear can transfer, the 225 mm section acts compositely for later loads.

---

## Sign convention used

Stress is taken as:

- **tension = positive**
- **compression = negative**

Sagging moment is taken as positive.

The stress formula used is:

\[
\sigma(y) = -\frac{P}{A} - \frac{P e y}{I} - \frac{M y}{I}
\]

where:

- \(P\) = prestress force (compression taken as positive magnitude)
- \(e\) = tendon eccentricity from the section centroid
- \(y\) = fiber coordinate measured from the section centroid, positive upward
- \(I\) = second moment of area
- \(M\) = sagging moment

---

## What this notebook gives you

- precast-only section properties
- composite section properties
- stage 1 moments and stresses
- stage 2 moments and incremental stresses
- final combined stresses at:
  - soffit of plank
  - top of precast / interface
  - top of topping
- interface shear flow \(q = VQ/I\)

You can tweak the concrete strengths, loads, prestress level, and propping assumptions.


In [47]:
# Cell 1 - imports and notebook magic
# Run this once. If handcalcs is not installed, uncomment the pip line.

# %pip install handcalcs

%load_ext handcalcs.render
import math
from dataclasses import dataclass



The handcalcs.render extension is already loaded. To reload it, use:
  %reload_ext handcalcs.render


In [48]:
from calc_setup import *
set_design_code('ec2_2004')


## Inputs

Edit these values first.  
The default loads below are placeholders for a typical building-floor check.


In [49]:
%%render params%%render params
# Parameters
# Geometry
b = 2400.0                 # mm, plank width
L = 4499.0                 # mm, span
h_p = 110.0                # mm, precast thickness
h_t = 115.0                # mm, topping thickness
h_tot = h_p + h_t          # mm, total composite thickness



<IPython.core.display.Latex object>

In [50]:
%%render params
# Prestress
n_strands = 8
A_ps_each = 54.8           # mm2 per 9.6/9.53 mm strand; edit to your actual strand data
A_ps = n_strands * A_ps_each
y_ps = 25.0                # mm from soffit to strand centroid; edit if required
f_pe_erection = 1050.0     # MPa, effective strand stress at erection / when arriving on site
delta_f_p_after_comp = 0.0 # MPa, further loss after topping hardens; keep 0 if ignored initially



<IPython.core.display.Latex object>

In [51]:
%%render params
# Concrete
gamma_precast = 24.0       # kN/m3
gamma_topping = 24.0       # kN/m3
Ec_precast = 34000.0       # MPa
Ec_topping = 30000.0       # MPa

# Construction-stage load sharing
# 1.0 = fully unpropped; 0.0 = fully propped
alpha_wet_to_precast = 1.0
alpha_construction_to_precast = 1.0

# Construction and service loads (surface loads)
q_construction = 1.5       # kN/m2 temporary construction load
q_sdl_after_comp = 1.0     # kN/m2 superimposed dead load applied after topping hardens
q_ll = 2.0                 # kN/m2 live load applied after topping hardens

<IPython.core.display.Latex object>

## Section properties

This cell calculates:

- precast-only centroid and inertia
- transformed composite centroid and inertia  
  using the modular ratio

\[
n = \frac{E_{c,\text{topping}}}{E_{c,\text{precast}}}
\]

If you want a simpler first pass, just set `Ec_topping = Ec_precast`.


In [52]:
%%render
# Cell 3 - section properties

# Precast-only section
A_p = b * h_p
ybar_p = h_p / 2
I_p = b * h_p**3 / 12

# Composite transformed section
n = Ec_topping / Ec_precast
A_t = b * h_t
A_t_tr = n * A_t

ybar_t = h_p + h_t / 2
ybar_c = (A_p * ybar_p + A_t_tr * ybar_t) / (A_p + A_t_tr)

I_t = b * h_t**3 / 12
I_c_tr = (    I_p + A_p * (ybar_c - ybar_p)**2    + n * (I_t + A_t * (ybar_t - ybar_c)**2))

# Fiber locations from soffit
y_soffit = 0.0
y_interface = h_p
y_top = h_tot

# Fiber coordinates relative to centroids
y_soffit_p = y_soffit - ybar_p
y_interface_p = y_interface - ybar_p

y_soffit_c = y_soffit - ybar_c
y_interface_c = y_interface - ybar_c
y_top_c = y_top - ybar_c

# Tendon eccentricities
e_p = y_ps - ybar_p
e_c = y_ps - ybar_c


<IPython.core.display.Latex object>

## Prestress force

Using:

\[
P_1 = A_{ps} f_{pe,\text{erection}}
\]

and later:

\[
\Delta P = A_{ps}\,\Delta f_{p,\text{after composite}}
\]

with MPa = N/mm², so force comes out in **N**.


In [53]:
%%render
# Cell 4 - prestress force

P1_N = A_ps * f_pe_erection
P1_kN = P1_N / 1000

delta_P_N = A_ps * delta_f_p_after_comp
delta_P_kN = delta_P_N / 1000


<IPython.core.display.Latex object>

## Line loads on the plank

For the actual **2.4 m wide plank**, the line loads are:

\[
w = q \times b
\]

with \(b\) in metres for the load conversion.


In [54]:
%%render
# Cell 5 - line loads

b_m = b / 1000
L_m = L / 1000
h_p_m = h_p / 1000
h_t_m = h_t / 1000

w_self_precast = gamma_precast * b_m * h_p_m
w_wet_topping = gamma_topping * b_m * h_t_m
w_construction = q_construction * b_m

w_stage1 = (    w_self_precast    + alpha_wet_to_precast * w_wet_topping    + alpha_construction_to_precast * w_construction)

w_sdl_after_comp = q_sdl_after_comp * b_m
w_ll_after_comp = q_ll * b_m
w_stage2 = w_sdl_after_comp + w_ll_after_comp


<IPython.core.display.Latex object>

## Stage 1 — precast only

Assuming simple support for the erection stage:

\[
M_1 = \frac{w_1 L^2}{8}
\qquad
V_1 = \frac{w_1 L}{2}
\]

Stresses in the **precast plank only** are then:

\[
\sigma_1(y) = -\frac{P_1}{A_p} - \frac{P_1 e_p y}{I_p} - \frac{M_1 y}{I_p}
\]

where \(y\) is measured from the **precast centroid**.


In [55]:
%%render
# Cell 6 - stage 1 actions and stresses

M1_kNm = w_stage1 * L_m**2 / 8
V1_kN = w_stage1 * L_m / 2

M1_Nmm = M1_kNm * 1e6

sigma_soffit1 = -(P1_N / A_p) - (P1_N * e_p * y_soffit_p / I_p) - (M1_Nmm * y_soffit_p / I_p)
sigma_interface1 = -(P1_N / A_p) - (P1_N * e_p * y_interface_p / I_p) - (M1_Nmm * y_interface_p / I_p)


<IPython.core.display.Latex object>

## Stage 2 — composite section

After the topping hardens and the interface can transfer shear, later loads act on the composite section.

\[
M_2 = \frac{w_2 L^2}{8}
\qquad
V_2 = \frac{w_2 L}{2}
\]

Incremental stresses are:

\[
\Delta \sigma(y) = -\frac{\Delta P}{A_c^\*}
- \frac{\Delta P e_c y}{I_c^\*}
- \frac{M_2 y}{I_c^\*}
\]

This notebook uses the **transformed section** quantities \(A_c^\*\) and \(I_c^\*\).

For a first-pass check with similar concrete grades, the transformed and gross values will be very close.


In [56]:
%%render
# Cell 7 - transformed composite area and stage 2 actions

A_c_tr = A_p + A_t_tr

M2_kNm = w_stage2 * L_m**2 / 8
V2_kN = w_stage2 * L_m / 2

M2_Nmm = M2_kNm * 1e6

delta_sigma_soffit = -(delta_P_N / A_c_tr) - (delta_P_N * e_c * y_soffit_c / I_c_tr) - (M2_Nmm * y_soffit_c / I_c_tr)
delta_sigma_interface = -(delta_P_N / A_c_tr) - (delta_P_N * e_c * y_interface_c / I_c_tr) - (M2_Nmm * y_interface_c / I_c_tr)
delta_sigma_top = -(delta_P_N / A_c_tr) - (delta_P_N * e_c * y_top_c / I_c_tr) - (M2_Nmm * y_top_c / I_c_tr)


<IPython.core.display.Latex object>

## Final combined stresses

At final service stage:

- **precast fibers** receive Stage 1 stress **plus** Stage 2 increment
- **topping fibers** receive only the Stage 2 increment, because the topping was not present in Stage 1


In [57]:
%%render
# Cell 8 - final service stresses

sigma_final_soffit = sigma_soffit1 + delta_sigma_soffit
sigma_final_interface_precast_side = sigma_interface1 + delta_sigma_interface
sigma_final_top_of_topping = delta_sigma_top


<IPython.core.display.Latex object>

## Interface shear flow

For composite action, check horizontal shear transfer at the interface:

\[
q = \frac{VQ}{I}
\]

Using the topping area above the interface:

\[
Q = (n A_t)\,(y_t - y_c)
\]

This gives **shear flow** \(q\) in **N/mm**.

If you want interface shear stress:

\[
\tau = \frac{q}{b_i}
\]

For this plank, \(b_i \approx 2400 \, \text{mm}\) unless you are checking a reduced effective interface width.


In [58]:
%%render
# Cell 9 - interface shear flow under stage 2 loading only

Q_interface = A_t_tr * (ybar_t - ybar_c)

q_interface_N_per_mm = (V2_kN * 1000) * Q_interface / I_c_tr
tau_interface_MPa = q_interface_N_per_mm / b


<IPython.core.display.Latex object>

## Optional: total-service line load and moment on composite section

Some engineers also like to see the **total final service moment** on the member:

\[
M_{serv,total} = \frac{(w_1 + w_2)L^2}{8}
\]

This is fine for reporting the final action effect,  
but the **stress calculation still needs the staged split** already done above.


In [59]:
%%render
# Cell 10 - reporting values

w_total_service = w_stage1 + w_stage2
M_total_service_kNm = w_total_service * L_m**2 / 8
V_total_service_kN = w_total_service * L_m / 2


<IPython.core.display.Latex object>

## Notes for use

### 1. If the slab is propped during topping
Set:

```python
alpha_wet_to_precast = 0.0
alpha_construction_to_precast = 0.0
```

or a partial fraction if the props take only part of the load.

### 2. If you want same-modulus gross composite properties
Set:

```python
Ec_topping = Ec_precast
```

### 3. If your strand centroid is not 25 mm from soffit
Edit:

```python
y_ps = ...
```

### 4. If your strand area differs from 54.8 mm² each
Edit:

```python
A_ps_each = ...
```

### 5. Units
- geometry: mm
- stress: MPa
- line load: kN/m
- moment: kN·m
- interface shear flow: N/mm

---

## Interpretation

For positive bending:

- the **topping mainly adds compression depth and stiffness**
- the **PT strands remain low in the precast**
- the **composite effect enters through the new centroid and inertia**
- the **topping only participates after it hardens**

If you want, the next notebook cell I can add is an **EC2-style ULS flexural capacity check** for the 225 mm composite section using the bonded prestressing strands at the bottom.
